# 1 Load Data

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("../../Data/student_performance_data.csv")
df.drop(columns=['overall_score','student_id'], inplace=True)
df

,gender,study_hours_per_day,attendance_percentage,assignment_score,midterm_score,final_exam_score,participation_score,internet_access,extra_classes,parent_education,sleep_hours,grade
0,Male,4.54,69.98,36.47,70.70,53.10,17.96,Yes,No,Master,8.09,D
1,Female,5.26,84.80,34.25,27.92,87.17,11.29,No,Yes,Bachelor,4.73,D
2,Male,8.69,73.76,72.29,70.92,99.61,76.10,No,Yes,PhD,8.73,B
3,Male,4.06,45.00,97.63,31.73,88.85,33.55,No,No,Bachelor,8.22,C
4,Male,8.83,51.13,65.19,78.28,54.23,88.99,No,No,Bachelor,8.59,C
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,Female,6.86,59.65,60.41,39.51,66.43,44.97,Yes,Yes,High School,8.56,C
9996,Male,2.60,83.62,62.45,48.96,81.40,45.11,Yes,Yes,Bachelor,4.21,C
9997,Female,1.46,95.40,67.08,51.51,87.58,65.49,Yes,No,Bachelor,4.72,B
9998,Female,7.15,78.24,97.73,46.51,75.49,61.21,No,No,Bachelor,5.28,B


In [2]:
df['grade'].value_counts()

grade
C    5073
B    2704
D    2008
A     154
F      61
Name: count, dtype: int64

In [3]:
df = df[df['grade'].isin(['C','B'])]

In [4]:
df.columns

Index(['gender', 'study_hours_per_day', 'attendance_percentage',
       'assignment_score', 'midterm_score', 'final_exam_score',
       'participation_score', 'internet_access', 'extra_classes',
       'parent_education', 'sleep_hours', 'grade'],
      dtype='str')

##### ohe -> 'gender'
##### ordinal -> 'parent_education'
##### Label -> 'internet_access', 'extra_classes'
##### scaling_transform -> 'study_hours_per_day', 'attendance_percentage','assignment_score', 'participation_score',midterm_score', 'final_exam_score', 'sleep_hours'



In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder, StandardScaler, PowerTransformer

scaling_transform = Pipeline([
    ("scl", StandardScaler()),
    ('power', PowerTransformer())
])

preprocessor = ColumnTransformer(transformers=[
    ("ohe", OneHotEncoder(sparse_output=False,handle_unknown='ignore'),['gender']),
    ("ord", OrdinalEncoder(categories=[['High School','Bachelor','Master','PhD']]), ['parent_education']),
    ("ord_bin", OrdinalEncoder(), ['internet_access', 'extra_classes']),
    ("scaling_transform", scaling_transform, ['study_hours_per_day', 'attendance_percentage','assignment_score', 'midterm_score', 'final_exam_score','participation_score', 'sleep_hours'])
    
],remainder='passthrough')

In [6]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:,:-1], df.iloc[:,-1], test_size=0.2, random_state=42)

In [7]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [8]:
lb = LabelEncoder()
y_train = lb.fit_transform(y_train)
y_test = lb.transform(y_test)

In [9]:
X_train

array([[ 1.        ,  0.        ,  2.        , ...,  1.55723615,
        -0.75892707, -0.89485911],
       [ 0.        ,  1.        ,  0.        , ...,  0.72436296,
        -0.29935487,  0.29754975],
       [ 1.        ,  0.        ,  0.        , ...,  1.39260359,
         1.18104473, -0.11537298],
       ...,
       [ 0.        ,  1.        ,  1.        , ..., -1.84484122,
         1.65800829,  0.10459476],
       [ 0.        ,  1.        ,  0.        , ..., -0.78651245,
        -0.41800821, -1.37117388],
       [ 1.        ,  0.        ,  2.        , ...,  1.00338957,
         1.44685614, -1.19442421]], shape=(6221, 12))

In [10]:
import torch
X_train = torch.from_numpy(X_train.astype(np.float32))
X_test = torch.from_numpy(X_test.astype(np.float32))
y_train = torch.from_numpy(np.asarray(y_train).astype(np.float32))   # Series - > Dataframe
y_test = torch.from_numpy(np.asarray(y_test).astype(np.float32))

# 3. Build Model

In [11]:
import torch.nn as nn
class MySimplePerceptron(nn.Module):
    def __init__(self, X_train):
        super().__init__()
        
        self.network = nn.Sequential(
            nn.Linear(X_train.shape[1], 6),
            nn.ReLU(),
            
            nn.Linear(6,3),
            nn.ReLU(),
            
            nn.Linear(3, 1),
            nn.Sigmoid()
        )
        
    def forward(self, X_train):
        return self.network(X_train)
    

# 4. Train Model

In [12]:
def train_model(learning_rate = 0.1,epochs = 100):
    model = MySimplePerceptron(X_train)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    
    loss_function = nn.BCELoss()
    
    for epoch in range(epochs):
        # forward propagation
        y_pred = model.forward(X_train)
        
        # loss calculate
        loss = loss_function(y_pred, y_train.view(-1,1))
        
        # reinitialize gradient
        optimizer.zero_grad()
        
        # backpropagation
        loss.backward()
        
        # update weight and bias
        optimizer.step()
        
        print(f"Epoch: {epoch+1}, Loss: {loss.item()}")
        
    return model

In [13]:
model = train_model(epochs=40, learning_rate=0.01)

Epoch: 1, Loss: 0.6512343883514404
Epoch: 2, Loss: 0.6494889855384827
Epoch: 3, Loss: 0.647497296333313
Epoch: 4, Loss: 0.6450117826461792
Epoch: 5, Loss: 0.641901433467865
Epoch: 6, Loss: 0.6381595134735107
Epoch: 7, Loss: 0.6336713433265686
Epoch: 8, Loss: 0.6283563375473022
Epoch: 9, Loss: 0.6221204400062561
Epoch: 10, Loss: 0.6150953769683838
Epoch: 11, Loss: 0.60741126537323
Epoch: 12, Loss: 0.5992171764373779
Epoch: 13, Loss: 0.5907227993011475
Epoch: 14, Loss: 0.5820743441581726
Epoch: 15, Loss: 0.573274552822113
Epoch: 16, Loss: 0.5641531348228455
Epoch: 17, Loss: 0.5545570254325867
Epoch: 18, Loss: 0.5442603230476379
Epoch: 19, Loss: 0.5332117676734924
Epoch: 20, Loss: 0.5215227603912354
Epoch: 21, Loss: 0.5093688368797302
Epoch: 22, Loss: 0.49698710441589355
Epoch: 23, Loss: 0.48469874262809753
Epoch: 24, Loss: 0.472698837518692
Epoch: 25, Loss: 0.4609283208847046
Epoch: 26, Loss: 0.4492539167404175
Epoch: 27, Loss: 0.4375530481338501
Epoch: 28, Loss: 0.42582014203071594
Epoc

# 5. Evaluate Model

In [14]:
def evaluate_model(threshold=0.5):
    with torch.no_grad():  
        y_pred = model.forward(X_test)
        y_pred = (y_pred > threshold).float()
        
        accuracy = (y_test == y_pred).float().mean()
        print(f"Accuracy: {accuracy}")

In [15]:
evaluate_model(threshold=0.5)

Accuracy: 0.5479775071144104


In [16]:
y_train

tensor([0., 0., 0.,  ..., 1., 1., 0.])